# 07 — Clasificación de Texto
**Goal:** Construir un pipeline completo de clasificación multiclase: del texto crudo a la predicción.

Tarea: clasificar tickets de soporte en 6 categorías automáticamente.

```
texto → TfidfVectorizer → clasificador → etiqueta de categoría
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay,
    f1_score, accuracy_score
)

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

STOPWORDS_ES = [
    'a','al','algo','ante','como','con','de','del','desde','el','ella','en',
    'entre','es','esta','este','esto','fue','la','las','le','lo','los','me',
    'mi','muy','no','nos','o','para','pero','por','que','se','si','sin',
    'su','sus','también','te','todo','un','una','y','ya','yo','tuve','llevo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
topic_templates = {
    'app_ux': [
        'La app está muy lenta y tarda en cargar la pantalla',
        'La aplicación no carga, sigue dando error al abrir',
        'La app se cierra sola cuando intento iniciar sesión',
        'Muy lenta la aplicación, tarda demasiado en responder',
        'La interfaz de la app es confusa y difícil de usar',
        'No puedo entrar a la app, siempre carga pero no abre',
    ],
    'otp_seguridad': [
        'No me llegó el OTP al celular para confirmar',
        'El código OTP no llega por SMS, ya intenté varias veces',
        'El código de verificación no aparece en mi teléfono',
        'Necesito el OTP pero no llega ningún mensaje al celular',
        'El SMS con el código tarda demasiado o no llega nunca',
        'No recibo el código de seguridad para iniciar sesión',
    ],
    'documentos': [
        'La carga de documentos falla constantemente en la app',
        'No puedo subir los documentos requeridos, siempre falla',
        'El proceso de carga de documentos da error todo el tiempo',
        'Los documentos no se suben correctamente al sistema',
        'Intenté cargar mi documento de identidad tres veces sin éxito',
        'El sistema no acepta mi archivo de documento de identidad',
    ],
    'credito_evaluacion': [
        'Llevo días esperando la evaluación crediticia sin respuesta',
        'Mi solicitud de crédito lleva una semana en evaluación',
        'No hay información sobre el estado de mi evaluación crediticia',
        'Quiero saber cuándo aprueban mi solicitud de tarjeta',
        'La evaluación de crédito no termina nunca, mucho tiempo',
        'Por qué rechazaron mi solicitud de crédito sin explicación',
    ],
    'soporte_atencion': [
        'El call center no responde, llevo horas esperando',
        'Nadie del soporte responde mis mensajes ni llamadas',
        'Pésima atención al cliente, nadie resuelve mis problemas',
        'Llamé cuatro veces y siguen sin resolver mi caso',
        'El chat de soporte no funciona y el teléfono no contesta',
        'El agente no sabía nada y no pudo resolver mi problema',
    ],
    'experiencia_positiva': [
        'Excelente servicio, aprobaron mi tarjeta muy rápido',
        'El proceso de solicitud fue fácil y muy eficiente',
        'Increíbles beneficios, recomiendo la tarjeta sin dudarlo',
        'Muy contento con el servicio, todo salió perfecto',
        'La activación fue sencilla y el proceso muy rápido',
        'Proceso ágil y atención excelente, totalmente recomendado',
    ],
}

rng = np.random.default_rng(42)
corpus, labels = [], []
for label, templates in topic_templates.items():
    n = 150
    for _ in range(n):
        corpus.append(templates[rng.integers(len(templates))])
        labels.append(label)

df = pd.DataFrame({'text': corpus, 'category': labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset: {len(df)} documentos | {df.category.nunique()} categorías')
print(df.category.value_counts())

## 1. Split train/test

In [ ]:
X = df['text']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')

## 2. Comparar clasificadores

In [ ]:
TFIDF = TfidfVectorizer(
    preprocessor=preprocess,
    stop_words=STOPWORDS_ES,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
)

classifiers = [
    ('Logistic Regression',    LogisticRegression(max_iter=1000, C=1.0, multi_class='multinomial')),
    ('LinearSVC',              LinearSVC(max_iter=2000, C=1.0)),
    ('Complement Naive Bayes', ComplementNB()),
    ('Random Forest',          RandomForestClassifier(n_estimators=200, random_state=42)),
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

print(f'{"Modelo":<28} {"CV F1 macro":>12}  {"Test F1 macro":>14}  {"Test Acc":>9}')
print('-' * 70)

pipes = {}
for name, clf in classifiers:
    pipe = Pipeline([('tfidf', TFIDF), ('clf', clf)])
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring='f1_macro', n_jobs=-1).mean()
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    test_f1  = f1_score(y_test, y_pred, average='macro')
    test_acc = accuracy_score(y_test, y_pred)
    print(f'{name:<28} {cv_f1:12.4f}  {test_f1:14.4f}  {test_acc:9.4f}')
    pipes[name] = pipe
    results.append({'model': name, 'cv_f1': cv_f1, 'test_f1': test_f1})

## 3. Reporte detallado del mejor modelo

In [ ]:
best_name = max(results, key=lambda x: x['test_f1'])['model']
best = pipes[best_name]
y_pred = best.predict(X_test)

print(f'Mejor modelo: {best_name}\n')
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=[l.replace('_', '\n') for l in sorted(y.unique())],
    cmap='Blues', ax=ax, xticks_rotation=45
)
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

## 4. Top features por categoría

In [ ]:
# Solo funciona con modelos que tienen coef_
lr_pipe = pipes['Logistic Regression']
lr_clf  = lr_pipe.named_steps['clf']
fn      = lr_pipe.named_steps['tfidf'].get_feature_names_out()
classes = lr_clf.classes_

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
colors = ['#4361ee','#f72585','#06d6a0','#ff9f1c','#7209b7','#e63946']

for i, (cls, coef_row) in enumerate(zip(classes, lr_clf.coef_)):
    top = np.argsort(coef_row)[::-1][:10]
    axes[i].barh(fn[top][::-1], coef_row[top][::-1], color=colors[i])
    axes[i].set_title(cls.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Coef.')

plt.suptitle('Top 10 features por categoría — Logistic Regression', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Predicción en producción

In [ ]:
nuevos = [
    'La aplicación no carga y me da error cuando intento entrar',
    'No recibí el SMS con el código para verificar mi identidad',
    'Quiero saber por qué no aprobaron mi solicitud de crédito',
    'Excelente experiencia, el proceso fue muy rápido y simple',
    'El soporte no me responde hace tres días, nadie me ayuda',
    'No pude subir mi cédula de identidad, el sistema da error',
]

# Usar el mejor modelo
preds = best.predict(nuevos)
if hasattr(best.named_steps['clf'], 'predict_proba'):
    probs = best.predict_proba(nuevos).max(axis=1)
    prob_label = 'Confianza'
else:
    probs = [None] * len(nuevos)
    prob_label = ''

print(f'{"Ticket":<55} {"Categoría predicha":<22} {prob_label}')
print('-' * 90)
for text, pred, prob in zip(nuevos, preds, probs):
    prob_str = f'{prob:.3f}' if prob is not None else ''
    print(f'{text:<55} {pred:<22} {prob_str}')

## Resumen
| Modelo | Cuándo usar |
|---|---|
| `LogisticRegression` | Default; interpretable, buenas probabilidades |
| `LinearSVC` | Más rápido en vocab grande, sin `predict_proba` nativo |
| `ComplementNB` | Desbalance de clases; muy rápido |
| `RandomForest` | Robusto pero más lento con texto esparso |

| Técnica | Impacto |
|---|---|
| `ngram_range=(1,2)` | Captura frases → mejora F1 en texto corto |
| `sublinear_tf=True` | Reduce dominancia de términos muy frecuentes |
| `stratify=y` | Mantiene proporción de clases en split |
| `f1_macro` | Métrica correcta cuando las clases son balanceadas |

**Siguiente:** `08_real_world.ipynb` — pipeline NLP completo aplicado a datos de crecimiento.